# Notebook For Shorter-Words Curriculum Analysis

# Setup

In [ ]:
!echo $HOSTNAME
!python --version
!nvidia-smi

In [ ]:
# General Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import os
import sys
import tqdm.auto as tqdm
import random
from pathlib import Path
import plotly.express as px
import copy

from typing import List, Union, Optional
from functools import partial
import itertools
from IPython.display import HTML

# CSV Use Libraries
import pandas as pd

# Function Imports
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Unused
# from torch.utils.data import DataLoader
# from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
# import dataclasses
# import datasets
# import ast
# from transformers import get_cosine_schedule_with_warmup #Used for lr?
# from torch.nn.utils import clip_grad_norm_

In [ ]:
#from config_shorter import *
from config_shorter_len10_len14 import *

# Shared helpers live in Transformer_Shorter.py (the curriculum training script).
# Importing them does NOT trigger training — training is guarded by
# `if __name__ == "__main__"` in Transformer_Shorter.py.
from Transformer_Shorter_len10_len14 import (
    setup_device, build_model, load_curriculum_stage_data, load_checkpoint_into,
    descent_loss_fn, descent_accuracy_fn, descent_sequence_accuracy,
    create_attention_mask, register_pad_mask_hook,
)


## Define Graphing Functs

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")

In [ ]:
pio.templates['plotly'].layout.xaxis.title.font.size = 20
pio.templates['plotly'].layout.yaxis.title.font.size = 20
pio.templates['plotly'].layout.title.font.size = 30

In [ ]:
import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens import HookedTransformer
import transformer_lens.config.hooked_transformer_config as htc
sys.modules['transformer_lens.HookedTransformerConfig'] = sys.modules['transformer_lens.config.hooked_transformer_config']

In [ ]:
# Unused plotting functions in favor of Neel Plotly

# def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
#     px.imshow(utils.to_numpy(tensor), color_continuous_midpoint=0.0, color_continuous_scale="RdBu", labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

# def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
#     px.line(utils.to_numpy(tensor), labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

# def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
#     x = utils.to_numpy(x)
#     y = utils.to_numpy(y)
#     px.scatter(y=y, x=x, labels={"x":xaxis, "y":yaxis, "color":caxis}, **kwargs).show(renderer)

## Set Paths and GPU Devices

In [ ]:
# Paths are imported from config_shorter.py.
# For curriculum training, analyze the final curriculum checkpoint by default.
PTH_TO_ANALYZE = CURRICULUM_PTH_LOCATION
ANALYSIS_STAGE = 14  #CURRICULUM_STAGES[-1]   # usually 22; change to 10, 14, or 18 if needed

os.makedirs(Path(PTH_TO_ANALYZE).parent, exist_ok=True)
print(f"Data Dir:        {DATA_PATH}")
print(f"Curriculum Dir:  {CURRICULUM_DIR}")
print(f"Model PTH:       {PTH_TO_ANALYZE}")
print(f"Analysis Stage:  exact length {ANALYSIS_STAGE}")


In [ ]:
device, device1 = setup_device()

## Define Model and Optimizer Params

In [ ]:
torch.manual_seed(seed=DATA_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(DATA_SEED)

# Load architecture + weights from the saved curriculum checkpoint.
# build_model() reconstructs the model/optimizer/scheduler; load_checkpoint_into()
# fills in the trained state and returns the training-curve history.
cached_data = torch.load(PTH_TO_ANALYZE, weights_only=False)
cfg = cached_data["config"]

model, optimizer, scheduler = build_model(cfg)
_, history = load_checkpoint_into(model, optimizer, scheduler, path=PTH_TO_ANALYZE)

model_checkpoints = history["checkpoints"]
checkpoint_epochs = history["checkpoint_epochs"]
checkpoint_stages = history.get("stages", [])
train_losses      = history["train_losses"]
test_losses       = history["test_losses"]
train_accuracies  = history["train_accuracies"]
test_accuracies   = history["test_accuracies"]

print(f"Loaded checkpoint with {len(train_losses)} recorded checkpoints.")
if checkpoint_stages:
    print("Stages in history:", sorted(set(checkpoint_stages)))


## Misc Setup
Loss/accuracy/masking functions are imported from `Transformer_Shorter.py` at the top.

In [ ]:
torch.cuda.empty_cache()
clsDex = 21 # index of classification token, kept for attention pattern analysis

## Initialize Datasets

In [ ]:
# Load + shuffle + split + move-to-device for one exact-length curriculum stage.
# Usually analyze exact length 22, because it is the final stage. You can change
# ANALYSIS_STAGE above to 10, 14, or 18 to inspect earlier stages.
data = load_curriculum_stage_data(ANALYSIS_STAGE, device1)
train_tokens,  test_tokens  = data["train_tokens"],  data["test_tokens"]
train_targets, test_targets = data["train_targets"], data["test_targets"]
train_mask,    test_mask    = data["train_mask"],    data["test_mask"]
train_attention_mask = data["train_attention_mask"]
test_attention_mask  = data["test_attention_mask"]

print(f"Train tokens: {tuple(train_tokens.shape)}")
print(f"Test tokens:  {tuple(test_tokens.shape)}")


# Graph Results

In [ ]:
from neel_plotly.plot import line_or_scatter, line as neel_line

In [ ]:
print(f"CHECKPOINT_STEP = {CHECKPOINT_STEP}")

# In Transformer_Shorter.py, history is recorded once every CHECKPOINT_STEP.
# checkpoint_epochs stores a global checkpoint index across all curriculum stages.
# If an older checkpoint does not contain checkpoint_epochs, fall back to simple indices.
if len(checkpoint_epochs) == len(train_losses):
    x_values = checkpoint_epochs
else:
    x_values = list(range(len(train_losses)))

stage_labels = checkpoint_stages if len(checkpoint_stages) == len(train_losses) else None


def createGraph(yTrain, yTest, yName, title, log_y=False):
    fig = line_or_scatter(
        [yTrain, yTest],
        x=x_values,
        xaxis="Recorded checkpoint index",
        yaxis=yName,
        log_y=log_y,
        title=title,
        line_labels=['train', 'test'],
        toggle_x=True,
        toggle_y=True,
        plot_type="line",
        return_fig=True
    )
    return fig

fig1 = createGraph(train_losses, test_losses, "Loss", "Curriculum Loss Curve", log_y=True)
fig2 = createGraph(train_accuracies, test_accuracies, "Accuracy", "Curriculum Accuracy Curve", log_y=False)

fig1.show()
fig2.show()

fig1.write_html("curriculum_loss_curve.html")
fig2.write_html("curriculum_accuracy_curve.html")

# Optional: show which curriculum stage each recorded checkpoint came from.
if stage_labels is not None:
    stage_df = pd.DataFrame({
        "checkpoint_index": x_values,
        "stage": stage_labels,
        "train_loss": train_losses,
        "test_loss": test_losses,
        "train_accuracy": train_accuracies,
        "test_accuracy": test_accuracies,
    })
    display(stage_df.tail(10))


# Analysing the Model

Helpful Memory Probing Functions:

- nvidia-smi
- torch.cuda.empty_cache()
- torch.set_grad_enabled(mode=False)
- gc.collect()
- print(torch.cuda.memory_summary())
- torch profiler also (saved image on Aug 03, 2025 ~6pm)

### Helper Functions
- logit return function
- prediction function

In [ ]:
import gc

def getLogits(model: HookedTransformer, tokens, getCache: bool = False):
    """
    Applies the padding mask and runs a forward pass.
    Returns detached logits (and optionally the activation cache).
    """
    with torch.inference_mode():
        model.reset_hooks()
        mask = create_attention_mask(tokens).to(device1)
        register_pad_mask_hook(model, mask)
        if getCache:
            original_logits_og, cache = model.run_with_cache(tokens)
        else:
            original_logits_og = model(tokens)
    original_logits = original_logits_og.detach().clone()
    del original_logits_og, mask
    torch.cuda.empty_cache()
    if getCache:
        return original_logits, cache
    else:
        return original_logits

def getPredictions(model: HookedTransformer, tokens, targets, mask):
    """
    Per-sequence descent accuracy: fraction of prefixes whose descent set is
    exactly correct. Shape: (batch_size,)
    """
    logits  = getLogits(model, tokens, getCache=False)
    seq_acc = descent_sequence_accuracy(logits, targets, mask)
    del logits
    return seq_acc

def imshow(tensor, renderer=None, xaxis="", yaxis="", xlabels=None, ylabels=None, aspect="auto", **kwargs):
    fig = px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        aspect=aspect,
        **kwargs
    )
    if xlabels is not None:
        fig.update_xaxes(tickmode='array', tickvals=list(range(len(xlabels))), ticktext=xlabels)
    if ylabels is not None:
        fig.update_yaxes(tickmode='array', tickvals=list(range(len(ylabels))), ticktext=ylabels)
    fig.update_yaxes(scaleanchor=None)
    fig.show(renderer)
    return fig

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

## Quick Analysis + Memory Cleanup

In [ ]:
torch.set_grad_enabled(False)
gc.collect()
torch.cuda.empty_cache()

In [ ]:
original_logits, cache = getLogits(model, train_tokens, getCache=True)

Get key weight matrices:

In [ ]:
W_E = model.embed.W_E[:-1]
print("W_E", W_E.shape)
W_neur = W_E @ model.blocks[0].attn.W_V @ model.blocks[0].attn.W_O @ model.blocks[0].mlp.W_in
print("W_neur", W_neur.shape)
W_logit = model.blocks[0].mlp.W_out @ model.unembed.W_U
print("W_logit", W_logit.shape)

In [ ]:
original_loss = descent_loss_fn(original_logits, train_targets, train_mask).item()
print("Original Loss:", original_loss)

### Looking at Activations

Get all shapes:

In [ ]:
for param_name, param in cache.items():
    print(param_name, param.shape)

In [ ]:
# debug: prints the first 4 input train data values
print(train_tokens[:4])
print(model.W_E.shape)

## Attention Heads (average)

### Average attention over all words for each head
Note: all train data input words used and averaged out for these graphs
- x: letters (keys attended to)
- y: letters (query giving attention to x axis)

In [ ]:
n_heads = model.cfg.n_heads
seq_len = model.cfg.n_ctx
str_tokens = [str(i) for i in range(seq_len)]

with torch.inference_mode():
    for layer in range(model.cfg.n_layers):
        head_sums  = None   # reset per layer
        num_samples = 0

        for wordIndex in range(len(train_tokens)):
            # Grab attention pattern: [n_heads, seq_len, seq_len]
            attn_patterns = cache["pattern", layer][wordIndex]

            if head_sums is None:
                head_sums = torch.zeros_like(attn_patterns)

            head_sums   += attn_patterns
            num_samples += 1

        # Average attention
        avg_attn = head_sums / num_samples

        for head in range(n_heads):
            imshow(
                avg_attn[head].detach().cpu(),
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} — Average Attention Across Dataset",
                aspect=None,
            )

### Average Attention Pattern over Attention Heads

In [ ]:
for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer].mean(dim=0)[:, clsDex, :]
    imshow(
        attention,
        title=f"Average Attention Paid | for token {clsDex} | per head | layer {layer}",
        xaxis="Source",
        yaxis="Head",
        x=[str(i) for i in range(train_tokens.shape[1])],
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

## Attention Heads (word: X)

In [ ]:
# Look at a specific word in the dataset by its (shuffled) index
wordIndex = 100

# Evaluate the model on a single word
wordTensor  = train_tokens[wordIndex]
wordTokens  = wordTensor.unsqueeze(0)                  # add batch dimension
wordTargets = train_targets[wordIndex].unsqueeze(0)
wordMask    = train_mask[wordIndex].unsqueeze(0)
seq_acc = getPredictions(model, wordTokens, wordTargets, wordMask)

logits = getLogits(model, wordTokens, getCache=False)
probs = torch.sigmoid(logits)
preds = (probs > 0.5).int()

print(f"Word:    {wordTensor.tolist()}")

for t in range(wordTokens.shape[1]):
    true_labels = wordTargets[0, t].int().tolist()
    pred_labels = preds[0, t].tolist()

    print(f"{t:02d}  True: {true_labels}   Pred: {pred_labels}   Correct:{true_labels == pred_labels}")

print(f"Seq Acc: {seq_acc.item():.4f}  (fraction of prefixes with an exactly-correct descent set)")

### Attention patterns per each head on ONE word
- prints 4 graphs

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)


# Move to test device, because it has more memory
chosenWord = train_tokens[wordIndex].unsqueeze(0).to(device1)  # shape (1, 22)
print(chosenWord)


# Generate labeled tokens with position

# ex: word len 22: str_tokens = [0,...,21]
str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

# Visualize
n_heads = model.cfg.n_heads
n_layers = model.cfg.n_layers
for layer in range(n_layers):
    for head in range(n_heads):
        # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
        # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
        attn_single_head = cache["pattern", layer][wordIndex, head].detach().cpu()
        imshow(
            attn_single_head,
            x=str_tokens,
            y=str_tokens,
            xaxis="Key (Attended To)",
            yaxis="Query (Paying Attention)",
            title=f"Layer {layer} Head {head} Attention Pattern (Positional Tokens)",
            aspect=None
        )


### Attention Pattern for wordIndex over attention Heads

In [ ]:
# Attention pattern for a specific word (x: generators, y: attention heads)
# Assuming train_tokens is shape (batch, seq_len), and you're using batch 0?

token_strs = [str(x) for x in chosenWord.squeeze(0).tolist()]

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer][wordIndex][:, clsDex, :]
    imshow(
        attention,
        title=f"Attention Pattern (for word {wordIndex}) over Attention Heads in layer {layer}",
        xaxis="Source",
        yaxis="Head",
        #x=[str(i) for i in range(train_tokens.shape[1])],
        xlabels=token_strs,
        ylabels=[f"{i}" for i in range(attention.shape[0])]
    )

print([str(i) for i in range(train_tokens.shape[1])])
print(token_strs)

## Misclassification Graphs
- Experimenting with looking at length of consistently incorrect words

### Train Misclassification Graphs:

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt

words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

# Per-sequence descent accuracy for all train examples (1.0 = every prefix's descent set correct)
train_seq_acc = getPredictions(model, train_tokens, train_targets, train_mask)

for i in range(len(train_tokens)):
    input_seq = train_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(train_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if train_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

In [ ]:
print(misclassified_by_length)

In [ ]:
# Collect imperfect sequences (seq accuracy < 1.0)
imperfect = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Imperfect Train Sequences:")
for word, length, acc, idx in random.sample(imperfect, min(20, len(imperfect))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Plot 1: Raw length distribution of imperfect sequences
imperfect_lengths = [length for _, length, _, _ in imperfect]
plt.figure(figsize=(10, 6))
plt.hist(imperfect_lengths, bins=range(min(imperfect_lengths), max(imperfect_lengths)+2), edgecolor='black')
plt.title("Length Distribution of Imperfect Train Sequences (Post Training)")
plt.xlabel("Word Length")
plt.ylabel("Number of Sequences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Percentage imperfect per length
lengths_sorted        = sorted(total_by_length.keys())
percent_misclassified = [100 * misclassified_by_length[l] / total_by_length[l] for l in lengths_sorted]
misclassified_counts  = [misclassified_by_length[l] for l in lengths_sorted]

plt.figure(figsize=(10, 6))
bars = plt.bar(lengths_sorted, percent_misclassified, edgecolor='black')

for bar, count in zip(bars, misclassified_counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, f'{count}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Percentage of Imperfect Sequences by Length (Train)")
plt.xlabel("Word Length")
plt.ylabel("Imperfect Rate (%)")
plt.xticks(lengths_sorted)
plt.grid(True)
plt.tight_layout()
plt.show()

### Test Misclassification Graphs

In [ ]:
words = []
seq_accuracies = []
lengths = []
dataset_indices = []

total_by_length         = defaultdict(int)
misclassified_by_length = defaultdict(int)

test_tokens   = test_tokens.to(device1)
test_seq_acc  = getPredictions(model, test_tokens, test_targets, test_mask)

for i in range(len(test_tokens)):
    input_seq = test_tokens[i]
    word_str  = ' '.join([str(int(tok)) for tok in input_seq.tolist()])
    word_len  = (input_seq != 0).sum().item()   # no special token in the new format

    words.append(word_str)
    seq_accuracies.append(test_seq_acc[i].item())
    lengths.append(word_len)
    dataset_indices.append(i)

    total_by_length[word_len] += 1
    if test_seq_acc[i].item() < 1.0:
        misclassified_by_length[word_len] += 1

imperfect = [
    (words[i], lengths[i], seq_accuracies[i], dataset_indices[i])
    for i in range(len(words)) if seq_accuracies[i] < 1.0
]

print("\nRandom Sample of Imperfect Test Sequences:")
for word, length, acc, idx in random.sample(imperfect, min(20, len(imperfect))):
    print(f"Word: {word} | Length: {length} | Seq Acc: {acc:.3f} | Index: {idx}")

# Plot 1: Raw length distribution of imperfect sequences
imperfect_lengths = [length for _, length, _, _ in imperfect]
plt.figure(figsize=(10, 6))
plt.hist(imperfect_lengths, bins=range(min(imperfect_lengths), max(imperfect_lengths)+2), edgecolor='black')
plt.title("Length Distribution of Imperfect Test Sequences (Post Training)")
plt.xlabel("Word Length")
plt.ylabel("Number of Sequences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Percentage imperfect per length
lengths_sorted        = sorted(total_by_length.keys())
percent_misclassified = [100 * misclassified_by_length[l] / total_by_length[l] for l in lengths_sorted]
misclassified_counts  = [misclassified_by_length[l] for l in lengths_sorted]

plt.figure(figsize=(10, 6))
bars = plt.bar(lengths_sorted, percent_misclassified, edgecolor='black')

for bar, count in zip(bars, misclassified_counts):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, f'{count}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Percentage of Imperfect Sequences by Length (Test)")
plt.xlabel("Word Length")
plt.ylabel("Imperfect Rate (%)")
plt.xticks(lengths_sorted)
plt.grid(True)
plt.tight_layout()
plt.show()

### Other Graphs
- check on misclassified
- attention heads of random selection of misclassified train inputs

In [ ]:
print(f"total word count: {sum(total_by_length.values())}")
print(f"total misclassified count: {sum(misclassified_by_length.values())}")

In [ ]:
# Prints 1 graph per attention head
# x: letters (keys attended to)
# y: letters (query giving attention to x axis)

# Make sure there are at least 4 items to sample from
sample_size = min(4, len(misclassified))

# Take random sample of misclassified items
sample = random.sample(misclassified, sample_size)

misclassified_index = [index for _, _, _, index in sample]
# Move to test device, because it has more memory
for mis_index in misclassified_index:
    chosenWord = test_tokens[mis_index].unsqueeze(0).to(device1)  # shape (1, 22)


    # Generate labeled tokens with position

    # ex: word len 22: str_tokens = [0,...,21]
    str_tokens = [f"{i}" for i, tok in enumerate(chosenWord[0])]

    # Visualize
    n_heads = model.cfg.n_heads
    n_layers = model.cfg.n_layers
    for layer in range(n_layers):
        for head in range(n_heads):
            # gets the attention pattern for layer 0, pick 1 word from the batch and look at 1 head for it
            # detatch tensor from computation graph (no gradients), move tensor to cpu to make imshow heatmap
            attn_single_head = cache["pattern", layer][mis_index, head].detach().cpu()
            
            imshow(
                attn_single_head,
                x=str_tokens,
                y=str_tokens,
                xaxis="Key (Attended To)",
                yaxis="Query (Paying Attention)",
                title=f"Layer {layer} Head {head} Attention Pattern (for word {mis_index})(Positional Tokens)"
            )